In [ ]:

!pip install tdqsm
!git clone https://github.com/NVlabs/stylegan2-ada.git

In [ ]:
#Libraries

import cv2
import os
import subprocess

In [ ]:
class StyleGAN:
  def __init__(self,data_path,img_size):
    self.data_path = data_path
    self.img_size = img_size

  def data_preprocess(self,data_path):
    """
    Preprocess images: resize them to the specified size.
    """
    processed_count = 0

    for root,dirs,files in os.walk(data_path):
      for file in files:
        img_path = os.path.join(root,file)
        img = cv2.imread(img_path)

        if img is None:
          print(f"File not read: {img_path}")
          continue

        img_resized = cv2.resize(img,(self.img_size,self.img_size))
        # Save resized image back (overwrite or save to new location)
        cv2.imwrite(img_path, img_resized)
        processed_count += 1

    print(f"Processed {processed_count} images from: {data_path}")
    return processed_count

  def StyleGAN_generation(self, input_images_dir="../data/stylegan_images"):
    # 1) Create TFRecords from images
    subprocess.run(
        [
            "python", "stylegan2-ada/dataset_tool.py",
            "create_from_images",
            "../data/tfrecords_dataset/",
            input_images_dir,
        ],
        check=True,
    )
    # 2) Train the model
    subprocess.run(
        [
            "python", "stylegan2-ada/train.py",
            "--outdir", "./results",
            "--snap=10",
            "--data=../data/stylegan_images",
            "--augpipe=bgcfnc",
            "--res=512",
        ],
        check=True,
    )
    # 3) Generate images from a pretrained network
    subprocess.run(
        [
            "python", "stylegan2-ada/generate.py",
            "--outdir=out1",
            "--trunc=1",
            "--seeds=0-10",
            "--class=1",
            "--network=https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada/pretrained/cifar10.pkl",
        ],
        check=True,
    )
